# DiabeticLens — Model Evaluation and Tuning

This notebook evaluates the machine learning models trained in Notebook 03.

The evaluation focuses on:
- Confusion Matrix
- Classification Report
- ROC-AUC
- Precision-Recall Curve
- Cross-Validation
- Hyperparameter Tuning
- Threshold Analysis
- Final Model Selection

The goal is to select a suitable diabetes prediction model while considering
the class imbalance in the dataset.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier
)

In [ ]:
df = pd.read_csv("../data/diabetes.csv")

print("Original dataset shape:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

In [ ]:
df = pd.read_csv("../data/diabetes.csv")

print("Original dataset shape:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

In [ ]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
preprocessor = joblib.load("../models/preprocessor.pkl")

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training data:", X_train_processed.shape)
print("Processed testing data:", X_test_processed.shape)

In [ ]:
logistic_model = joblib.load("../models/logistic_regression.pkl")
decision_tree = joblib.load("../models/decision_tree.pkl")
random_forest = joblib.load("../models/random_forest.pkl")
knn = joblib.load("../models/knn.pkl")
gradient_boosting = joblib.load("../models/gradient_boosting.pkl")

print("All saved models loaded successfully.")

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "KNN": knn,
    "Gradient Boosting": gradient_boosting
}

predictions = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test_processed)

print("Predictions generated successfully.")

In [ ]:
for name, model in models.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    
    print(
        classification_report(
            y_test,
            predictions[name],
            digits=4
        )
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes = axes.flatten()

for i, (name, pred) in enumerate(predictions.items()):
    
    cm = confusion_matrix(y_test, pred)
    
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=axes[i]
    )
    
    axes[i].set_title(name)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

# Hide unused subplot
axes[-1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

for name, model in models.items():
    
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    
    auc_score = roc_auc_score(y_test, y_prob)
    
    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

for name, model in models.items():
    
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    
    auc_score = roc_auc_score(y_test, y_prob)
    
    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc_score:.4f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.grid()
plt.show()

In [ ]:
evaluation_results = []

for name, model in models.items():
    
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    
    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df.round(4)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-Fold Stratified Cross-Validation created.")

In [ ]:
gb_cv_scores = cross_val_score(
    gradient_boosting,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print("Gradient Boosting F1 scores:")
print(gb_cv_scores)

print("\nMean F1:", gb_cv_scores.mean())
print("Standard Deviation:", gb_cv_scores.std())

In [ ]:
rf_cv_scores = cross_val_score(
    random_forest,
    X_train_processed,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print("Random Forest F1 scores:")
print(rf_cv_scores)

print("\nMean F1:", rf_cv_scores.mean())
print("Standard Deviation:", rf_cv_scores.std())

In [ ]:
param_dist = {
    "n_estimators": [100, 150, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

print("Hyperparameter search space created.")

In [ ]:
param_dist = {
    "n_estimators": [100, 150, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

print("Hyperparameter search space created.")